In [1]:
import pandas as pd
import numpy as np
import html, re, os

reviews = pd.read_json("data/raw/All_Beauty.jsonl", lines=True, convert_dates=False)

reviews["date"] = pd.to_datetime(reviews["timestamp"], unit="ms")

log = []
def log_step(step, data):
    log.append({"step": step,
                "reviews": len(data),
                "products": data["parent_asin"].nunique(),
                "users": data["user_id"].nunique()})

log_step("Raw file", reviews)
print(len(reviews))

701528


In [2]:
reviews = reviews.sort_values(["parent_asin", "timestamp", "user_id"]).reset_index(drop=True)

reviews = reviews.drop_duplicates(subset=["user_id", "parent_asin", "text"], keep="first")
log_step("Exact duplicates removed", reviews)

reviews = reviews.drop_duplicates(subset=["user_id", "parent_asin"], keep="first")
log_step("Repeat reviews removed", reviews)

pd.DataFrame(log)

,step,reviews,products,users
0,Raw file,701528,112565,631986
1,Exact duplicates removed,694228,112565,631986
2,Repeat reviews removed,693929,112565,631986


In [3]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)
    text = re.sub(r"<br\s*/?>", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

reviews["text_clean"] = reviews["text"].apply(clean_text)
reviews["title_clean"] = reviews["title"].apply(clean_text)

reviews["n_words"] = reviews["text_clean"].str.split().str.len()

reviews[["text", "text_clean", "n_words"]].head()

,text,text_clean,n_words
0,great,great,1
1,Good product,Good product,2
2,Makes my hair look healthy,Makes my hair look healthy,5
3,I exercise a lot being older you don't smell ...,I exercise a lot being older you don't smell t...,18
4,La manera de entrega poca seguridad de que lle...,La manera de entrega poca seguridad de que lle...,13


In [4]:
reviews = reviews.sort_values(["parent_asin", "timestamp", "user_id"]).reset_index(drop=True)

reviews["n_prior"] = reviews.groupby("parent_asin").cumcount()

prior_sum = reviews.groupby("parent_asin")["rating"].cumsum() - reviews["rating"]

reviews["prior_mean"] = prior_sum / reviews["n_prior"]

reviews.loc[reviews["n_prior"] < 5, "prior_mean"] = np.nan

previous = reviews.groupby("parent_asin")["rating"].shift(1)
reviews["prior_mean_last10"] = (previous.groupby(reviews["parent_asin"])
                                .rolling(10, min_periods=10).mean()
                                .reset_index(level=0, drop=True))

reviews[["parent_asin",  "date", "rating", "n_prior", "prior_mean", "prior_mean_last10"]].head(30)

,parent_asin,date,rating,n_prior,prior_mean,prior_mean_last10
0,0005946468,2018-02-02 04:54:01.775,5,0,NaN,NaN
1,0123034892,2017-09-26 17:25:51.406,5,0,NaN,NaN
2,0124784577,2019-06-24 23:29:59.244,5,0,NaN,NaN
3,0124784577,2019-09-01 01:23:54.115,5,1,NaN,NaN
4,0124784577,2019-09-09 14:59:29.877,3,2,NaN,NaN
5,0515059560,2014-09-17 02:00:11.000,4,0,NaN,NaN
6,0692508988,2016-03-24 21:27:06.000,5,0,NaN,NaN
7,069267599X,2016-10-21 16:55:20.000,5,0,NaN,NaN
8,069267599X,2016-12-02 16:42:57.000,5,1,NaN,NaN
9,069267599X,2017-02-16 19:14:35.000,5,2,NaN,NaN


In [5]:
row = reviews[reviews["prior_mean"].notna()].sample(1, random_state=42).iloc[0]
product = reviews[reviews["parent_asin"] == row["parent_asin"]]
earlier = product[product["n_prior"] < row["n_prior"]]["rating"]

print("Earlier ratings:", list(earlier))
print("Their average by hand:", round(earlier.mean(), 3))
print("Code's average:       ", round(row["prior_mean"], 3))

check = (reviews.groupby("parent_asin")["rating"]
         .transform(lambda s: s.shift(1).expanding().mean()))
check[reviews["n_prior"] < 5] = np.nan

has_avg = reviews["prior_mean"].notna()
assert (check.notna() == has_avg).all(), "Different reviews have an average - there is a bug!"
assert np.allclose(check[has_avg], reviews.loc[has_avg, "prior_mean"]), "Mismatch found - there is a bug!"
assert reviews["prior_mean"].dropna().between(1, 5).all()
print("All", has_avg.sum(), "averages match the second calculation and use only earlier reviews")

Earlier ratings: [2, 3, 2, 2, 1, 3, 1, 5, 1, 1, 3, 2, 3, 2, 3, 3, 5, 1, 1]
Their average by hand: 2.316
Code's average:        2.316
All 410866 averages match the second calculation and use only earlier reviews


In [6]:
reviews = reviews.sort_values(["user_id", "timestamp", "parent_asin"])

reviews["user_prior_reviews"] = reviews.groupby("user_id").cumcount()

reviews["log_experience"] = np.log1p(reviews["user_prior_reviews"])

reviews["repeat_reviewer"] = reviews["user_prior_reviews"] > 0

reviews = reviews.sort_values(["parent_asin", "timestamp", "user_id"]).reset_index(drop=True)

reviews["repeat_reviewer"].value_counts()

repeat_reviewer
False    631986
True      61943
Name: count, dtype: int64

In [7]:
meta = pd.read_json("data/raw/meta_All_Beauty.jsonl", lines=True, convert_dates=False)
meta = meta[["parent_asin", "title", "store", "price"]]

meta = meta.rename(columns={"title": "product_title"})

meta["price"] = pd.to_numeric(meta["price"], errors="coerce")

print("Each product appears once in meta:", meta["parent_asin"].is_unique)

rows_before = len(reviews)
reviews = reviews.merge(meta, on="parent_asin", how="left")
rows_after = len(reviews)

print("Rows before:", rows_before, "| Rows after:", rows_after)
assert rows_before == rows_after

reviews["log_price"] = np.log(reviews["price"].where(reviews["price"] > 0))

log_step("3. Product info joined (D1, D6)", reviews)
reviews[["parent_asin", "product_title", "store", "price", "log_price"]].head()

Each product appears once in meta: True
Rows before: 693929 | Rows after: 693929


,parent_asin,product_title,store,price,log_price
0,0005946468,Patanjali Moisturizer Cream - Pack of 2,patanjali,NaN,NaN
1,0123034892,Yardley Lace Satin Deodorant Spray for women 1...,None,NaN,NaN
2,0124784577,WOW Organics Apple Cider Vinegar Shampoo - 300 mL,WOW Organics Apple Cider Vinegar Shampoo - 300 mL,NaN,NaN
3,0124784577,WOW Organics Apple Cider Vinegar Shampoo - 300 mL,WOW Organics Apple Cider Vinegar Shampoo - 300 mL,NaN,NaN
4,0124784577,WOW Organics Apple Cider Vinegar Shampoo - 300 mL,WOW Organics Apple Cider Vinegar Shampoo - 300 mL,NaN,NaN


In [8]:
reviews["has_crowd"] = reviews["prior_mean"].notna()

reviews["dissent_neg"] = (reviews["rating"] <= 2) & (reviews["prior_mean"] >= 4)

reviews["agree_neg"] = (reviews["rating"] <= 2) & (reviews["prior_mean"] <= 2.5)

reviews["dissent_pos"] = (reviews["rating"] >= 4) & (reviews["prior_mean"] <= 2.5)

reviews["long_enough"] = reviews["n_words"] >= 25

reviews["post_chatgpt"] = reviews["date"] >= "2022-11-30"

reviews["age_days"] = (reviews["date"].max() - reviews["date"]).dt.days
reviews["product_n_reviews"] = reviews.groupby("parent_asin")["parent_asin"].transform("size")
reviews["any_helpful"] = reviews["helpful_vote"] >= 1

print("Reviews with 5+ earlier reviews:                    ", reviews["has_crowd"].sum())
print("Bad reviews of good products:                       ", reviews["dissent_neg"].sum())
print("Bad reviews of bad products:                        ", reviews["agree_neg"].sum())
print("Good reviews of bad products:                       ", reviews["dissent_pos"].sum())
print("Bad reviews of good products, 25+ words:            ", (reviews["dissent_neg"] & reviews["long_enough"]).sum())
print("Bad reviews of good products, 25+ words, with price:",
      (reviews["dissent_neg"] & reviews["long_enough"] & reviews["price"].notna()).sum())

Reviews with 5+ earlier reviews:                     410866
Bad reviews of good products:                        36247
Bad reviews of bad products:                         7866
Good reviews of bad products:                        3331
Bad reviews of good products, 25+ words:             16816
Bad reviews of good products, 25+ words, with price: 5566


In [9]:
log_step("4. Has a crowd of 5+ earlier reviews (D5)", reviews[reviews["has_crowd"]])
log_step("5. ...and 25+ words, for word features (D4)",
         reviews[reviews["has_crowd"] & reviews["long_enough"]])

cleaning_log = pd.DataFrame(log)
if os.path.exists("results/cleaning_log.csv"):
    print("cleaning_log.csv already exists - not overwritten.")
else:
    cleaning_log.to_csv("results/cleaning_log.csv", index=False)
print(cleaning_log)

keep_cols = ["parent_asin", "user_id", "date", "rating", "title_clean", "text_clean", "n_words",
             "helpful_vote", "any_helpful", "verified_purchase",
             "n_prior", "prior_mean", "prior_mean_last10",
             "user_prior_reviews", "log_experience", "repeat_reviewer",
             "product_title", "store", "price", "log_price",
             "has_crowd", "dissent_neg", "agree_neg", "dissent_pos",
             "long_enough", "post_chatgpt", "age_days", "product_n_reviews"]

if os.path.exists("data/processed/reviews_clean.parquet"):
    print("reviews_clean.parquet already exists - not overwritten.")
else:
    reviews[keep_cols].to_parquet("data/processed/reviews_clean.parquet", index=False)
    print("Saved:", len(reviews), "reviews")

cleaning_log.csv already exists - not overwritten.
                                          step  reviews  products   users
0                                     Raw file   701528    112565  631986
1                     Exact duplicates removed   694228    112565  631986
2                       Repeat reviews removed   693929    112565  631986
3              3. Product info joined (D1, D6)   693929    112565  631986
4    4. Has a crowd of 5+ earlier reviews (D5)   410866     22606  390400
5  5. ...and 25+ words, for word features (D4)   169384     18082  161063
reviews_clean.parquet already exists - not overwritten.
